# SmartStock AI - Exploratory Data Analysis (EDA)
This notebook performs a comprehensive exploratory data analysis on the mock sales dataset. 
It uses Plotly for interactive visualizations and generates insights to understand:
- Sales Trends & Seasonality
- Monthly & Weekly Demand
- Category & Store Performance
- Promotion & Holiday Impacts
- Price Elasticity & Inventory Analysis
- Feature Correlations & Distributions


## 1. Setup & Data Loading
First, we import the necessary libraries and load our dataset. We'll ensure the code is modular by defining helper functions for plotting.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Load dataset
df = pd.read_csv('../data/raw/mock_sales_data.csv')
df['Date'] = pd.to_datetime(df['Date'])

# Define a modular helper for plotting
def plot_time_series(data, x_col, y_col, title, color=None):
    fig = px.line(data, x=x_col, y=y_col, title=title, color=color, template='plotly_white')
    fig.update_layout(xaxis_title=x_col, yaxis_title=y_col)
    fig.show()

def plot_bar(data, x_col, y_col, title, color=None):
    fig = px.bar(data, x=x_col, y=y_col, title=title, color=color, template='plotly_white')
    fig.update_layout(xaxis_title=x_col, yaxis_title=y_col)
    fig.show()

df.head()


**Insight:** The dataset is successfully loaded. We have various features ranging from temporal dimensions (Date), categorical features (Store, Category), numerical performance metrics (Sales, Profit), to business drivers (Price, Promotion).

## 2. Sales Trends & Seasonality
Let's analyze how sales fluctuate over time.

In [ ]:
# Aggregate sales daily
daily_sales = df.groupby('Date')['Sales'].sum().reset_index()
plot_time_series(daily_sales, 'Date', 'Sales', 'Total Daily Sales Trend')

# Add moving average to smooth out noise
daily_sales['7-Day MA'] = daily_sales['Sales'].rolling(window=7).mean()
fig = px.line(daily_sales, x='Date', y=['Sales', '7-Day MA'], title='Daily Sales with 7-Day Moving Average', template='plotly_white')
fig.show()


**Insight:** 
- The overall sales trend shows regular fluctuations, likely driven by weekly cycles. 
- The 7-Day Moving Average smooths out the day-to-day variance and highlights broader macro trends and potential seasonal peaks. There appear to be periodic spikes representing high-demand events.

## 3. Monthly & Weekly Demand
Breaking down sales into month-over-month and week-over-week aggregates.

In [ ]:
# Extract time features
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.day_name()
# Sort days
days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

monthly_sales = df.groupby('Month')['Sales'].mean().reset_index()
plot_bar(monthly_sales, 'Month', 'Sales', 'Average Sales by Month')

weekly_sales = df.groupby('DayOfWeek')['Sales'].mean().reindex(days_order).reset_index()
plot_bar(weekly_sales, 'DayOfWeek', 'Sales', 'Average Sales by Day of Week')


**Insight:** 
- **Monthly Demand:** We can observe which months typically experience higher baseline demand. (e.g., Year-end holiday season or summer spikes).
- **Weekly Demand:** Weekends (Saturday/Sunday) exhibit noticeably higher average sales compared to weekdays, indicating weekend shopping behavior.

## 4. Category & Store Performance
Comparing how different product categories and individual stores perform.

In [ ]:
# Category performance
category_sales = df.groupby('Category')['Sales'].sum().reset_index().sort_values('Sales', ascending=False)
plot_bar(category_sales, 'Category', 'Sales', 'Total Sales by Product Category', color='Category')

# Store performance
store_sales = df.groupby('Store_ID')['Sales'].sum().reset_index().sort_values('Sales', ascending=False)
plot_bar(store_sales, 'Store_ID', 'Sales', 'Total Sales by Store')


**Insight:** 
- **Categories:** Certain categories heavily dominate the sales volume. Understanding the product mix is crucial for accurate forecasting.
- **Stores:** Store performance is relatively balanced/imbalanced. Identifying underperforming stores can help target operational improvements.

## 5. Promotion & Holiday Impact
Do promotions and holidays significantly drive sales?

In [ ]:
fig1 = px.box(df, x='Promotion', y='Sales', color='Promotion', title='Impact of Promotion on Sales', template='plotly_white')
fig1.show()

fig2 = px.box(df, x='Holiday', y='Sales', color='Holiday', title='Impact of Holidays on Sales', template='plotly_white')
fig2.show()


**Insight:** 
- **Promotions:** The median sales for periods with active promotions is significantly higher than non-promotional periods. Promotions are a strong driver of demand.
- **Holidays:** Sales on holidays also show elevated median values and higher variance, confirming that holidays inject temporary surges into the demand baseline.

## 6. Price Elasticity
Understanding how price variations affect sales volume.

In [ ]:
fig = px.scatter(df, x='Price', y='Sales', color='Category', opacity=0.6, title='Price vs. Sales Volume', template='plotly_white')
fig.show()

# Calculate rough elasticity proxy for a specific product
sample_product = df['Product_ID'].unique()[0]
prod_df = df[df['Product_ID'] == sample_product].sort_values('Date')
fig2 = px.scatter(prod_df, x='Discount', y='Sales', trendline='ols', title=f'Discount vs Sales for {sample_product}', template='plotly_white')
fig2.show()


**Insight:** 
- The scatter plot indicates a general downward trend: as price increases, sales volume tends to decrease, demonstrating standard price elasticity.
- The discount analysis shows a positive correlation between discount magnitude and sales volume, confirming that consumers respond strongly to price drops.

## 7. Inventory Analysis
Analyzing the relationship between inventory levels and stockouts (where Sales == Inventory).

In [ ]:
df['Stockout_Risk'] = np.where(df['Sales'] >= df['Inventory'], 'Stockout/High Risk', 'Safe')
stockout_counts = df['Stockout_Risk'].value_counts().reset_index()
stockout_counts.columns = ['Status', 'Count']
fig = px.pie(stockout_counts, values='Count', names='Status', title='Proportion of Potential Stockout Days', template='plotly_white')
fig.show()

fig2 = px.scatter(df, x='Inventory', y='Sales', color='Stockout_Risk', opacity=0.5, title='Sales vs Inventory (Identifying constraints)', template='plotly_white')
fig2.add_shape(type='line', x0=0, y0=0, x1=df['Inventory'].max(), y1=df['Inventory'].max(), line=dict(color='Red', dash='dash'))
fig2.show()


**Insight:** 
- The red dashed line (Sales = Inventory) represents the physical limit of sales due to stock constraints. 
- Points lying exactly on or near this line indicate days where demand likely exceeded supply, resulting in lost revenue. The pie chart quantifies the frequency of these risk events.

## 8. Correlation Matrix
Understanding multi-collinearity and relationships between numerical features.

In [ ]:
numeric_df = df.select_dtypes(include=[np.number]).drop(columns=['Month'])
corr = numeric_df.corr().round(2)

fig = px.imshow(corr, text_auto=True, aspect="auto", color_continuous_scale='RdBu_r', title='Feature Correlation Heatmap')
fig.show()


**Insight:** 
- Strong positive correlations exist between Profit and Sales, as well as Discount and Promotion.
- Weak or zero correlation is observed between weather variables (Temperature, Rainfall) and sales, suggesting weather might not be a primary driver unless interacting with specific categories.

## 9. Feature Distributions
Visualizing the distributions of key continuous variables.

In [ ]:
fig = make_subplots(rows=2, cols=2, subplot_titles=('Sales Distribution', 'Price Distribution', 'Temperature Distribution', 'Supplier Lead Time'))

fig.add_trace(go.Histogram(x=df['Sales'], name='Sales'), row=1, col=1)
fig.add_trace(go.Histogram(x=df['Price'], name='Price'), row=1, col=2)
fig.add_trace(go.Histogram(x=df['Temperature'], name='Temp'), row=2, col=1)
fig.add_trace(go.Histogram(x=df['Supplier_Lead_Time'], name='Lead Time'), row=2, col=2)

fig.update_layout(title_text='Distributions of Key Features', height=700, showlegend=False, template='plotly_white')
fig.show()


**Insight:** 
- **Sales:** Right-skewed distribution, typical for retail, meaning most days have average sales, but a few days (outliers/events) have massive spikes.
- **Price:** Fairly uniform/multi-modal depending on the categories present.
- **Supplier Lead Time:** Uniform distribution in the mock data, highlighting the variability in supply chain responsiveness.